In [ ]:
!nvidia-smi

## Load And Process Dataset

In [ ]:
from datasets import load_from_disk

# Load from disk (can load from hf hub @ loknezmomzter/pmc-patients-distill-5k)
data = load_from_disk("/mnt/huggingface/data/medgemma_extracts/train")

# Shuffle and split into train and eval sets
records = data.shuffle(seed=42)
split_records = records.train_test_split(test_size=0.1, seed=42)

train_records = split_records["train"]
test_records = split_records["test"]

print(f"len(train_set) - {len(train_records)}")
print(f"len(test_set) - {len(test_records)}")

In [ ]:

train_records[0]

In [ ]:
type(train_records)

### Prepare JSON Schema

In [ ]:
JSON_SCHEMA = {
    "summary": "A concise, 1-2 sentence abstractive summary of the clinical scenario.",
    "clinical_reasoning": "A step-by-step logical breakdown of the diagnoses, treatments, or clinical decisions made in the text. Explain WHY certain relationships exist. Keep short and brief but to the point.",
    "relationships": [
        {
            "subject": "Source entity (e.g., Patient, Drug, Symptom)",
            "predicate": "Use STANDARD POSITIVE relationships (e.g., HAS_HISTORY, SHOWS_SYMPTOM, DIAGNOSED_WITH, PRESCRIBED). Do not use negated verbs like 'DENIES' or 'LACKS'.",
            "object": "Target entity",
            "polarity": "positive OR negative (Use 'negative' if the patient denies the history or lacks the symptom)",
            "certainty": "confirmed, suspected, OR hedged",
            "evidence": "The exact verbatim text snippet that proves this relationship."
        }
    ],
    "keywords": ["List", "of", "important", "clinical", "NER", "terms"]
}

## Fine Tuning Setup And Model Config

In [ ]:
from unsloth.chat_templates import CHAT_TEMPLATES

for k in CHAT_TEMPLATES.keys():
    if "mistral" in k:
        print(k)

In [ ]:
from unsloth import FastVisionModel
from unsloth.chat_templates import get_chat_template


model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Ministral-3-3B-Instruct-2512-unsloth-bnb-4bit",
    max_seq_length=4096,
    # load_in_4bit=True, # Not required if the model is already 4bit-quantized
    use_gradient_checkpointing="unsloth"
)

model = FastVisionModel.get_peft_model(
    model,

    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_layers=True,
    finetune_mlp_layers=True,

    r=32,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    
    random_state=3407,
    use_rslora=False,
    loftq_config=False,

    target_modules="all-linear",
    modules_to_save=["lm_head", "embed_tokens"]
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="mistral"
)

In [ ]:
import json

def format_prompts(examples):
    instructions = examples["raw_medical_text"]
    outputs = examples["data"]
    texts = []

    schema_string = json.dumps(JSON_SCHEMA, indent=4)

    for instruction, output in zip(instructions, outputs):

        # Serialize to string if the column row is loaded as dict
        output_str = json.dumps(output) if isinstance(output, dict) else str(output)

        # Prepare chat messages array
        messages = [
            {
                "role": "system",
                "content": f"""
                You are an expert clinical informatician. Extract data strictly into this JSON schema:\n
                
                {schema_string}

                --------------------------
                CRITICAL FORMATTING RULES
                --------------------------
                1. Use ONLY double quotes (") for all JSON keys and string values.
                2. If clinical terms contain apostrophes (e.g., patient's, Alzheimer's), leave them as raw characters inside the double-quoted strings.
                3. Your entire response MUST start strictly with the '{{' character and end strictly with the '}}' character.
                4. Output raw JSON only. No explanations, no markdown formatting, no code blocks\n\n"""
            },
            {
                "role": "user",
                "content": f"""CONTEXT:\n{instruction}"""
            },
            {
                "role": "assistant",
                "content": output_str
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)

    return {"text": texts}

In [ ]:
# Transform the train and test sets for training
train_set = train_records.map(format_prompts, batched=True)
test_set = test_records.map(format_prompts, batched=True)


In [ ]:
train_set[0]

In [ ]:
import numpy as np

# Calculate the token length of the train set
token_counts = [len(tokenizer.tokenizer.encode(text)) for text in train_set["text"]]
p95 = np.percentile(token_counts, 95)

print(f"95% of your prompts are shorter than {p95} tokens")

In [ ]:
import torch
from trl import SFTTrainer, SFTConfig

# Initialize and configure SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,

    train_dataset=train_set,
    eval_dataset=test_set,

    args=SFTConfig(
        # Architecture config
        dataset_text_field="text",
        max_seq_length=4096,
        dataset_num_proc=2,

        # Memory and speed control
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        optim="adamw_8bit",
        bf16=torch.cuda.is_bf16_supported(),

        # Duration control
        max_steps=60,

        # Learning rate and scheduling
        learning_rate=2e-4,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,

        # Logging
        logging_steps=3,
        report_to="none",
        seed=3407,

        # Evaluation config
        per_device_eval_batch_size=2,
        eval_accumulation_steps=8,
        eval_strategy="steps",
        eval_steps=10,

        save_strategy="steps",
        save_steps=10,

        # Output config (mandatory argument)
        output_dir="/workspace/checkpoints",
        save_total_limit=3,

        # Finalizing the best model
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )
)

In [ ]:
# Programmatically extract the structural bounds of the tokenizer
sample_messages = [
    {"role": "user", "content": "<USER_CONTEXT>"},
    {"role": "assistant", "content": "<ASSISTANT_REPLY>"}
]
print(tokenizer.apply_chat_template(sample_messages, tokenize=False))

In [ ]:
from unsloth.chat_templates import train_on_responses_only

# Apply masking - prevent model from learning instructions
trainer = train_on_responses_only(
    trainer,
    instruction_part="[INST]",
    response_part="[/INST]"
)

In [ ]:
trainer_stats = trainer.train()

## Inference

In [2]:
import torch
import json
from datasets import load_from_disk
from unsloth import FastVisionModel
from unsloth.chat_templates import get_chat_template
from transformers import TextStreamer


model, tokenizer = FastVisionModel.from_pretrained(
    model_name="unsloth/Ministral-3-3B-Instruct-2512-unsloth-bnb-4bit",
    max_seq_length=4096,
)
FastVisionModel.for_inference(model)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="mistral"
)

print(f"\n✅ model and tokenizer initialized for inference\n")

# Initialize text streamer
streamer = TextStreamer(tokenizer, skip_prompt=True)

JSON_SCHEMA = {
    "summary": "A concise, 1-2 sentence abstractive summary of the clinical scenario.",
    "clinical_reasoning": "A step-by-step logical breakdown of the diagnoses, treatments, or clinical decisions made in the text. Explain WHY certain relationships exist. Keep short and brief but to the point.",
    "relationships": [
        {
            "subject": "Source entity (e.g., Patient, Drug, Symptom)",
            "predicate": "Use STANDARD POSITIVE relationships (e.g., HAS_HISTORY, SHOWS_SYMPTOM, DIAGNOSED_WITH, PRESCRIBED). Do not use negated verbs like 'DENIES' or 'LACKS'.",
            "object": "Target entity",
            "polarity": "positive OR negative (Use 'negative' if the patient denies the history or lacks the symptom)",
            "certainty": "confirmed, suspected, OR hedged",
            "evidence": "The exact verbatim text snippet that proves this relationship."
        }
    ],
    "keywords": ["List", "of", "important", "clinical", "NER", "terms"]
}

# Convert schema to string
schema_string = json.dumps(JSON_SCHEMA, indent=4)

system_prompt = f"""
You are an expert clinical informatician. Extract data strictly into this JSON schema:\n
        
{schema_string}

--------------------------
CRITICAL FORMATTING RULES
--------------------------
1. Use ONLY double quotes (") for all JSON keys and string values.
2. If clinical terms contain apostrophes (e.g., patient's, Alzheimer's), leave them as raw characters inside the double-quoted strings.
3. Your entire response MUST start strictly with the '{{' character and end strictly with the '}}' character.
4. Output raw JSON only. No explanations, no markdown formatting, no code blocks.\n\n
"""

# Load test set from disk and select one record at random
records = load_from_disk("/mnt/huggingface/data/medgemma_extracts/test") 
record = records.filter(lambda x: x["id"] == "7246274-1")[0]

print(f"\n✅ successfully loaded recrd {record['id']} from disk...\n")

# Extract relevant information from record
raw_medical_text = record["raw_medical_text"]
ground_truth = record["data"]
record_id = record["id"]

# Format the message using the same prompt as fine tuning
messages = [
    {
        "role": "system",
        "content": [
            {"type": "image"},
            {"type": "text", "text": system_prompt}
        ]
    },
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": f"""CONTEXT:\n{raw_medical_text}"""}
        ]
    },
]
print(f"\n⏳ generating output, please wait...\n")

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Apply chat template and move to GPU
inputs = tokenizer(
    [],
    text=[prompt],
    add_special_tokens=False,
    return_tensors="pt"
).to("cuda")

# Generate outputs
outputs = model.generate(
    **inputs,
    max_new_tokens=2048,
    use_cache=True,
    eos_token_id=tokenizer.eos_token_id,  # add this
    pad_token_id=tokenizer.eos_token_id,  # add this too
    streamer=streamer
)

# Decode and print response
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

==((====))==  Unsloth 2026.5.1: Fast Ministral3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 44.547 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]


✅ model and tokenizer initialized for inference


✅ successfully loaded recrd 7246274-1 from disk...


⏳ generating output, please wait...



TypeError: can only concatenate str (not "list") to str

In [ ]:
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print(response[0])

In [ ]:
print(ground_truth)

In [ ]:
print(raw_medical_text)